# Reliable SQL Reports

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_OER/blob/main/course_materials/notebooks/04_reliable_reporting.ipynb)
[View this notebook on GitHub](https://github.com/lolusername/CST4714_OER/blob/main/course_materials/notebooks/04_reliable_reporting.ipynb)

Day 1 counts tickets without multiplying them by events. Day 2 selects one latest event per ticket. Start with a fresh fixture: this notebook does not depend on the views or source_channel migration walkthrough. This is supplementary practice for Chapter 2, not an additional graded assignment.

Use a **Python 3 / CPU** Colab runtime. Run cells individually from the
top; pause at **Your work**. The database is temporary: download your
SQL before disconnecting or running cleanup. No cloud database account,
password, GPU, or notebook submission is required.

## Meet Metro Support Before Writing Queries

Metro Support is a fictional public-service help desk. All people and
records here are synthetic. A table's **grain** is what one row means.

| Table | One row means | Rows | Important columns |
|---|---|---:|---|
| `users` | one resident or staff member | 8 | `user_id`, `display_name` |
| `tickets` | one support request | 12 | `ticket_id`, `category`, `priority`, `status`, `assignee_id` |
| `ticket_events` | one recorded event on a ticket | 21 | `event_id`, `ticket_id`, `event_type`, `event_at` |

A ticket has one requester and an optional assignee, both referring to
`users.user_id`. An event refers to `tickets.ticket_id`. One ticket can
therefore have several event rows; an event is **not** another ticket.

**Active** means status `new`, `open`, or `in_progress`: 7 of the 12
tickets. Tickets **1004 and 1009** are active but have a `NULL` assignee
(not assigned yet). Do not drop them from a report. Every ticket in the
starting fixture has at least one event; future tickets might not.

The setup below contains the exact published
[8-user / 12-ticket / 21-event SQL fixture](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_03/materials/datasets/metro_support/postgres_setup.sql).
It is embedded here, so running the notebook does **not** fetch data
from GitHub. Read the `CREATE TABLE` definitions and a few `INSERT`
rows when you need to see where a result came from.

## Setup: A Real, Disposable PostgreSQL Database

In Colab, the next cell installs PostgreSQL with `apt-get` and starts
its local service. `sudo -u postgres` runs database commands as the
service's operating-system user, without a password. Package
installation needs internet access; the dataset does not.

**Local Jupyter only:** use an already-running disposable PostgreSQL
server, put its `bin` directory on `PATH`, and explicitly set `PGHOST`
to its Unix socket directory before running this cell. Set `PGPORT`
and `PGUSER` too if they differ from your local defaults. Your role
needs permission to create databases. Remote hosts are refused.
This notebook does not install, start, or stop a local computer's server.

The next two cells are Python setup, not SQL to submit. A unique database
name keeps the fixture's schema reset away from existing databases.
Do not replace that generated name with one of your own databases.

In [ ]:
import getpass
import os
from pathlib import Path
import subprocess
from uuid import uuid4

# Local Jupyter only: uncomment and supply your actual socket directory.
# os.environ["PGHOST"] = "/path/to/your/postgresql/socket"
# os.environ["PGPORT"] = "5432"

if "PRACTICE_DB" in globals():
    raise RuntimeError("Keep this database for both days, or run cleanup first.")
if os.environ.get("PGHOSTADDR") or os.environ.get("PGSERVICE"):
    raise RuntimeError("Unset PGHOSTADDR and PGSERVICE; use an explicit local socket.")

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "postgresql", "postgresql-client"], check=True)
    subprocess.run(["service", "postgresql", "start"], check=True)
    COLAB_SERVICE_STARTED = True

PGHOST = "/var/run/postgresql" if IN_COLAB else os.environ.get("PGHOST", "")
if not Path(PGHOST).is_absolute() or "," in PGHOST or not Path(PGHOST).is_dir():
    raise RuntimeError("Set PGHOST to one existing local Unix socket directory.")
PGPORT = "5432" if IN_COLAB else os.environ.get("PGPORT", "5432")
PGUSER = "postgres" if IN_COLAB else os.environ.get("PGUSER", getpass.getuser())
PG_PREFIX = ["sudo", "-u", "postgres"] if IN_COLAB else []
PG_ARGS = ["--host", PGHOST, "--port", PGPORT, "--username", PGUSER, "--no-password"]
subprocess.run(PG_PREFIX + ["pg_isready"] + PG_ARGS[:-1] + ["--dbname", "postgres"], check=True)
print("Local socket:", PGHOST, "Port:", PGPORT)

In [ ]:
# template0 supplies an empty database, not a copy of your course work.
if "PRACTICE_DB" in globals():
    raise RuntimeError("Database already created. Do not reset it between days.")
run_id = uuid4().hex
new_database = "cst4714_week04_" + run_id
subprocess.run(
    PG_PREFIX + ["createdb"] + PG_ARGS
    + ["--maintenance-db=postgres", "--template=template0", new_database],
    check=True,
)
PRACTICE_DB = new_database  # Remember only the database this run created.
print("This notebook owns:", PRACTICE_DB)

### One Small SQL Display Helper

`run_sql` sends the text between triple quotes to PostgreSQL's `psql`
program and prints its table output. Everything inside those quotes
is ordinary SQL. `-X` ignores personal psql settings; `ON_ERROR_STOP`
stops at the first error. `NULL` is printed explicitly, not as a blank.

Each call opens a **new connection**. Put `BEGIN`, every statement in
the transaction, and its `COMMIT` or `ROLLBACK` in **one call**. Do not
split a transaction or a temporary table across notebook cells.
Outside an explicit transaction, each statement commits separately.
On an error, the connection closes and any uncommitted transaction
rolls back; earlier committed work stays. Fix the first error and
rerun the complete intended block, not just its last line.

In [ ]:
def run_sql(sql_text):
    result = subprocess.run(
        PG_PREFIX + ["psql"] + PG_ARGS
        + ["-X", "--dbname", PRACTICE_DB, "--set=ON_ERROR_STOP=on",
           "--set=VERBOSITY=verbose", "--pset=null=NULL", "--pset=pager=off"],
        input=sql_text, text=True, capture_output=True,
    )
    print(result.stdout)
    if result.returncode:
        raise RuntimeError(result.stderr.strip())

### Load the Fixture Once

The published script starts with `DROP SCHEMA ... CASCADE`. Here it
runs **only in the new practice database** just printed above. The
guard prevents accidentally rerunning it over your work in this
notebook. Do not copy this reset into a real database or your submission.
The final result must show **8 users, 12 tickets, and 21 events**.

In [ ]:
if globals().get("FIXTURE_LOADED", False):
    raise RuntimeError("Fixture already loaded. Keep your work; do not reset between days.")
assert PRACTICE_DB == "cst4714_week04_" + run_id
setup_sql = """-- Metro Support PostgreSQL setup
-- Run only in a course or personal practice database. This resets the
-- metro_support schema so the dataset is reproducible.

DROP SCHEMA IF EXISTS metro_support CASCADE;
CREATE SCHEMA metro_support;
SET search_path TO metro_support, public;

CREATE TABLE users (
    user_id integer PRIMARY KEY,
    display_name text NOT NULL,
    email text NOT NULL UNIQUE,
    role text NOT NULL,
    neighborhood text NOT NULL,
    created_at timestamptz NOT NULL
);

CREATE TABLE tickets (
    ticket_id integer PRIMARY KEY,
    requester_id integer NOT NULL REFERENCES users(user_id),
    assignee_id integer REFERENCES users(user_id),
    category text NOT NULL,
    priority text NOT NULL,
    status text NOT NULL,
    subject text NOT NULL,
    opened_at timestamptz NOT NULL,
    closed_at timestamptz,
    CHECK (closed_at IS NULL OR closed_at >= opened_at)
);

CREATE TABLE ticket_events (
    event_id integer PRIMARY KEY,
    ticket_id integer NOT NULL REFERENCES tickets(ticket_id),
    actor_id integer NOT NULL REFERENCES users(user_id),
    event_type text NOT NULL,
    old_status text,
    new_status text,
    note text,
    event_at timestamptz NOT NULL
);

INSERT INTO users
    (user_id, display_name, email, role, neighborhood, created_at)
VALUES
    (101, 'Maya Chen', 'maya.chen@example.test', 'resident', 'Harbor', '2026-01-08T14:20:00Z'),
    (102, 'Luis Rivera', 'luis.rivera@example.test', 'resident', 'Northside', '2026-01-10T09:15:00Z'),
    (103, 'Amina Yusuf', 'amina.yusuf@example.test', 'resident', 'Central', '2026-01-12T18:05:00Z'),
    (104, 'Jordan Bell', 'jordan.bell@example.test', 'resident', 'Harbor', '2026-01-18T11:40:00Z'),
    (201, 'Priya Shah', 'priya.shah@example.test', 'agent', 'Central', '2025-11-03T13:00:00Z'),
    (202, 'Noah Williams', 'noah.williams@example.test', 'agent', 'Northside', '2025-11-05T13:00:00Z'),
    (203, 'Elena Garcia', 'elena.garcia@example.test', 'supervisor', 'Central', '2025-09-14T13:00:00Z'),
    (204, 'Sam Okafor', 'sam.okafor@example.test', 'analyst', 'Harbor', '2025-12-01T13:00:00Z');

INSERT INTO tickets
    (ticket_id, requester_id, assignee_id, category, priority, status, subject, opened_at, closed_at)
VALUES
    (1001, 101, 201, 'streetlight', 'high', 'open', 'Streetlight dark near bus stop', '2026-02-01T23:10:00Z', NULL),
    (1002, 102, 202, 'sanitation', 'medium', 'in_progress', 'Missed recycling pickup', '2026-02-02T15:45:00Z', NULL),
    (1003, 103, 201, 'water', 'urgent', 'resolved', 'Low water pressure', '2026-02-03T12:05:00Z', '2026-02-03T19:40:00Z'),
    (1004, 104, NULL, 'parks', 'low', 'new', 'Broken bench slat', '2026-02-04T17:20:00Z', NULL),
    (1005, 101, 202, 'sanitation', 'high', 'resolved', 'Overflowing corner bin', '2026-02-05T14:00:00Z', '2026-02-05T20:15:00Z'),
    (1006, 102, 201, 'streetlight', 'medium', 'in_progress', 'Flickering lamp outside library', '2026-02-06T01:30:00Z', NULL),
    (1007, 103, 202, 'parks', 'medium', 'open', 'Playground gate will not latch', '2026-02-07T16:10:00Z', NULL),
    (1008, 104, 201, 'water', 'high', 'resolved', 'Hydrant leaking slowly', '2026-02-08T10:25:00Z', '2026-02-09T09:05:00Z'),
    (1009, 101, NULL, 'transportation', 'medium', 'new', 'Bus shelter panel cracked', '2026-02-09T22:15:00Z', NULL),
    (1010, 102, 202, 'sanitation', 'low', 'closed', 'Replacement bin request', '2026-02-10T13:50:00Z', '2026-02-12T16:30:00Z'),
    (1011, 103, 201, 'transportation', 'high', 'open', 'Crosswalk signal delayed', '2026-02-11T08:35:00Z', NULL),
    (1012, 104, 202, 'streetlight', 'low', 'resolved', 'Lamp stays on during daytime', '2026-02-12T14:45:00Z', '2026-02-14T18:10:00Z');

INSERT INTO ticket_events
    (event_id, ticket_id, actor_id, event_type, old_status, new_status, note, event_at)
VALUES
    (5001, 1001, 101, 'created', NULL, 'open', 'Reported through mobile form', '2026-02-01T23:10:00Z'),
    (5002, 1001, 201, 'assigned', 'open', 'open', 'Electrical crew notified', '2026-02-02T14:05:00Z'),
    (5003, 1002, 102, 'created', NULL, 'open', 'Pickup was scheduled for Monday', '2026-02-02T15:45:00Z'),
    (5004, 1002, 202, 'status_changed', 'open', 'in_progress', 'Route supervisor checking vehicle log', '2026-02-03T13:30:00Z'),
    (5005, 1003, 103, 'created', NULL, 'open', 'Pressure lower on two floors', '2026-02-03T12:05:00Z'),
    (5006, 1003, 201, 'status_changed', 'open', 'in_progress', 'Crew dispatched', '2026-02-03T14:25:00Z'),
    (5007, 1003, 201, 'status_changed', 'in_progress', 'resolved', 'Valve adjustment restored pressure', '2026-02-03T19:40:00Z'),
    (5008, 1004, 104, 'created', NULL, 'new', 'Photo attached in original report', '2026-02-04T17:20:00Z'),
    (5009, 1005, 101, 'created', NULL, 'open', 'Bin blocks part of sidewalk', '2026-02-05T14:00:00Z'),
    (5010, 1005, 202, 'status_changed', 'open', 'resolved', 'Extra collection completed', '2026-02-05T20:15:00Z'),
    (5011, 1006, 102, 'created', NULL, 'open', 'Flicker repeats every few seconds', '2026-02-06T01:30:00Z'),
    (5012, 1006, 201, 'status_changed', 'open', 'in_progress', 'Ballast inspection scheduled', '2026-02-06T15:10:00Z'),
    (5013, 1007, 103, 'created', NULL, 'open', 'Gate opens toward play area', '2026-02-07T16:10:00Z'),
    (5014, 1008, 104, 'created', NULL, 'open', 'Small stream along curb', '2026-02-08T10:25:00Z'),
    (5015, 1008, 201, 'status_changed', 'open', 'resolved', 'Gasket replaced and area checked', '2026-02-09T09:05:00Z'),
    (5016, 1009, 101, 'created', NULL, 'new', 'No sharp edge visible', '2026-02-09T22:15:00Z'),
    (5017, 1010, 102, 'created', NULL, 'open', 'Current bin lid is missing', '2026-02-10T13:50:00Z'),
    (5018, 1010, 202, 'status_changed', 'open', 'closed', 'Replacement delivered', '2026-02-12T16:30:00Z'),
    (5019, 1011, 103, 'created', NULL, 'open', 'Wait exceeds one full light cycle', '2026-02-11T08:35:00Z'),
    (5020, 1012, 104, 'created', NULL, 'open', 'Possible photocell issue', '2026-02-12T14:45:00Z'),
    (5021, 1012, 202, 'status_changed', 'open', 'resolved', 'Photocell cleaned and tested', '2026-02-14T18:10:00Z');

-- Verification: these counts should be 8, 12, and 21.
SELECT 'users' AS table_name, count(*) AS row_count FROM users
UNION ALL
SELECT 'tickets', count(*) FROM tickets
UNION ALL
SELECT 'ticket_events', count(*) FROM ticket_events;
"""
run_sql(setup_sql)
FIXTURE_LOADED = True

### Read a Few Rows

`metro_support` is the schema (a namespace); `tickets` is the table.
`SELECT` chooses columns, `ORDER BY` makes display order predictable,
and `LIMIT 4` displays only four rows without removing stored data.
Expect ticket 1004 to show `NULL` for its assignee.

In [ ]:
run_sql("""
SELECT ticket_id, category, priority, status, assignee_id
FROM metro_support.tickets
ORDER BY ticket_id
LIMIT 4;
""")

## Day 1: Count at the Right Grain

Ticket 1003 has **three** event rows. Joining tickets directly to events
repeats that ticket three times. Inspect the rows before aggregating:
each output row below represents an event, not a separate ticket.

In [ ]:
run_sql("""
SELECT t.ticket_id, t.status, e.event_id, e.event_type
FROM metro_support.tickets AS t
LEFT JOIN metro_support.ticket_events AS e ON e.ticket_id = t.ticket_id
WHERE t.ticket_id = 1003
ORDER BY e.event_id;
""")

`COUNT(*)` counts input rows. This join produces **21 rows**, but there
are still only **12 distinct tickets**. A DISTINCT count helps diagnose
this example; it does not automatically repair other repeated measures,
such as summing a ticket's cost once for every event.

With a LEFT JOIN, a ticket with no event would still contribute one
joined row. `COUNT(e.event_id)` ignores that NULL event ID; `COUNT(*)`
does not. Our starting data has no zero-event ticket, so do not mistake
that accidental equality for a general rule.

In [ ]:
run_sql("""
SELECT count(*) AS joined_rows,
       count(DISTINCT t.ticket_id) AS distinct_tickets,
       count(e.event_id) AS recorded_events
FROM metro_support.tickets AS t
LEFT JOIN metro_support.ticket_events AS e ON e.ticket_id = t.ticket_id;
""")

### Worked Example: One Event Count per Ticket

`WITH event_counts AS (...)` gives a temporary name to a query result
for this **one statement**. This is a common table expression (CTE),
not a saved table or view. `GROUP BY ticket_id` reduces events to one
row per ticket before the join. `COUNT(*)` is correct **inside this CTE**
because it is counting rows of the events table.

Then LEFT JOIN that result to tickets **once**. At most one event-count
row matches each ticket, so ticket rows are not multiplied. `COALESCE`
takes the first non-NULL argument: missing counts become **0**, not a
missing ticket. `ORDER BY` controls the final display order.

In [ ]:
run_sql("""
WITH event_counts AS (
    SELECT ticket_id, count(*) AS event_count
    FROM metro_support.ticket_events
    GROUP BY ticket_id
)
SELECT t.ticket_id, t.status,
       COALESCE(ec.event_count, 0) AS event_count
FROM metro_support.tickets AS t
LEFT JOIN event_counts AS ec ON ec.ticket_id = t.ticket_id
ORDER BY t.ticket_id;
""")

Expect **12 rows**, one per ticket. In ticket-ID order 1001 through
1012, the event counts are **2, 2, 3, 1, 2, 2, 1, 2, 1, 2, 1, 2**.
They sum to **21**, matching the events table. The count for 1003 is 3;
the counts for unassigned tickets 1004 and 1009 are 1 each.

### Conditional Counts Without Removing Other Tickets

`COUNT(*) FILTER (WHERE condition)` feeds only matching rows into that
**one aggregate**. Other aggregates still see their own input. A query's
ordinary `WHERE` instead removes rows before **all** its aggregates.

This worked example summarizes the **whole tickets table**, not the
category report you will write. `AND` requires both conditions, while
`IN ('high', 'urgent')` accepts either priority. A high-priority resolved
ticket is not a high-or-urgent **active** ticket.

In [ ]:
run_sql("""
SELECT count(*) AS total_tickets,
       count(*) FILTER (
           WHERE status IN ('new', 'open', 'in_progress')
       ) AS active_tickets,
       count(*) FILTER (
           WHERE status IN ('new', 'open', 'in_progress')
             AND priority IN ('high', 'urgent')
       ) AS high_or_urgent_active_tickets
FROM metro_support.tickets;
""")

### Your Work: A Category Report, Not Another Per-Ticket Report

Write one row per `category` with **total_tickets**, **active_tickets**,
and **high_or_urgent_active_tickets**. These measures count **tickets**,
not events, so the tickets table alone is sufficient for the lab.
Use aggregate FILTER clauses rather than a whole-query active-only
WHERE. Do not submit the worked per-ticket report in place of the
category report.

If you include event counts in your report's intermediate result,
reuse the worked event-count CTE: LEFT JOIN it once to tickets with
`COALESCE(event_count, 0)`, then group the ticket-grain result by category.
Do not rejoin raw events after pre-aggregating them. This safe extension
is practice, not an additional required report or submission.

Self-check: **5 category rows**; the category totals sum to **12**,
active counts sum to **7**, and high-or-urgent active counts sum to
**2**. In every row, high-or-urgent active <= active <= total. Keep
categories even when they contain no active tickets. Write a separate
simple SELECT listing the two high-or-urgent active ticket IDs, and
compare those records with the embedded data. Explain in a SQL comment
what one report row represents and why the raw join's 21 rows were not
21 tickets.

Put your report and checks in **one SQL file**:
`week_04_category_report.sql`. The finished category query is your work.

In [ ]:
category_report_sql = """
-- TODO: Group tickets by category for the three requested counts.
-- TODO: Independently SELECT the two high-or-urgent active ticket IDs.
-- TODO: Include your checks and grain explanation as SQL comments.
"""
run_sql(category_report_sql)

### Optional: Download Your SQL, Not the Notebook

Finish `category_report_sql` above, including your SQL comments and checks.
This cell saves **exactly that string**, not every cell you ran and
not your query output. Review the printed text before submitting.
Set the switch to `True` when ready; it writes `week_04_category_report.sql` in the
runtime and opens Colab's download prompt. On local Jupyter it prints
the file's location. Repeating the export replaces that same file.
Downloading is not submitting; upload the SQL file to Brightspace.

In [ ]:
EXPORT_SQL = False  # Change to True only after finishing the workspace.
if EXPORT_SQL:
    if "-- TODO" in category_report_sql:
        raise ValueError("Replace the TODO placeholders with your own work first.")
    print(category_report_sql)  # Preview exactly what will be in the file.
    sql_file = Path("week_04_category_report.sql")
    sql_file.write_text(category_report_sql.strip() + "\n", encoding="utf-8")
    if IN_COLAB:
        from google.colab import files
        files.download(str(sql_file))
    else:
        print("Saved:", sql_file.resolve())

## Day 2: Pick One Latest Recorded Event

`MAX(event_at)` finds a time, not the complete event row associated
with it. If two events share that time, joining back on the maximum
can return two rows. A window function lets us number whole rows.

Read each part **before** filtering the numbered result:

| Clause | Meaning |
|---|---|
| `ROW_NUMBER()` | Assign consecutive numbers starting at 1; keep individual rows. |
| `OVER (...)` | Describe the related rows and their ordering for this calculation. |
| `PARTITION BY ticket_id` | Restart numbering for each ticket, not the entire events table. |
| `ORDER BY event_at DESC` | Put the greatest recorded timestamp first in that ticket. |
| `event_id DESC` | For equal timestamps, choose the greater unique event ID first. |
| `AS rn` | Name the calculated column so a later query can filter it. |

The inner ORDER BY controls numbering; a final ORDER BY controls
display. The tie rule is deterministic, **not evidence of real
chronological or business order**. A larger event ID need not mean
an event actually occurred later. See the PostgreSQL
[window-function tutorial](https://www.postgresql.org/docs/current/tutorial-window.html).

For ticket 1003, expect event IDs **5007, 5006, 5005** with rn **1, 2, 3**.
We display all numbered rows first, not just rn = 1.

In [ ]:
run_sql("""
SELECT ticket_id, event_id, event_type, event_at,
       ROW_NUMBER() OVER (
           PARTITION BY ticket_id
           ORDER BY event_at DESC, event_id DESC
       ) AS rn
FROM metro_support.ticket_events
WHERE ticket_id = 1003
ORDER BY rn;
""")

### Temporary Equal-Timestamp Demonstration

These **two invented rows** exist only in a temporary demonstration
table. Both deliberately have the same timestamp. Expect event 9902
to receive rn = 1 and 9901 to receive rn = 2 because of the ID tie rule.
Without that second key, tied rows have no guaranteed relative order,
even if repeated runs happen to look stable.

Do not add artificial time offsets to the real data to manufacture a
business order. These timestamps teach a tie, not historical truth.
If true business order matters, obtain a trustworthy source ordering
field or clarify that the source cannot establish it. This block rolls
back; it changes none of the published 21 events.

In [ ]:
run_sql("""
BEGIN;
CREATE TEMP TABLE event_order_demo (
    ticket_id integer, event_id integer, event_at timestamptz
);
INSERT INTO event_order_demo VALUES
    (1001, 9901, '2026-02-02T14:05:00Z'),
    (1001, 9902, '2026-02-02T14:05:00Z');

SELECT ticket_id, event_id, event_at,
       ROW_NUMBER() OVER (
           PARTITION BY ticket_id
           ORDER BY event_at DESC, event_id DESC
       ) AS rn
FROM event_order_demo
ORDER BY rn;
ROLLBACK;
""")

### Worked Example: Latest Event for All 12 Tickets

The first CTE numbers the events. The second CTE reads that result
and filters `rn = 1`. This extra query level matters: a WHERE in the
same SELECT that computes ROW_NUMBER cannot filter its new `rn` value.

LEFT JOIN the resulting **at-most-one row per ticket** to **all tickets**.
Do not place `rn = 1` in the final WHERE after that LEFT JOIN: it would
remove tickets without an event. Preserve NULL event details instead
of making up an event or timestamp. `t.status` is the ticket's stored
current status, not a status inferred from an arbitrary event.

The lab uses an equivalent variation: LEFT JOIN `ranked_events AS e`
directly with `ON e.ticket_id = t.ticket_id AND e.rn = 1`. Our second
CTE filters before joining; that ON condition filters matches while
joining. Both preserve tickets with no matching event. Neither should
move the event-rank condition into the final WHERE.

In [ ]:
run_sql("""
WITH ranked_events AS (
    SELECT ticket_id, event_id, event_type, event_at,
           ROW_NUMBER() OVER (
               PARTITION BY ticket_id
               ORDER BY event_at DESC, event_id DESC
           ) AS rn
    FROM metro_support.ticket_events
),
latest_event AS (
    SELECT ticket_id, event_id, event_type, event_at
    FROM ranked_events
    WHERE rn = 1
)
SELECT t.ticket_id, t.status, le.event_id, le.event_type, le.event_at
FROM metro_support.tickets AS t
LEFT JOIN latest_event AS le ON le.ticket_id = t.ticket_id
ORDER BY t.ticket_id;
""")

Expect **12 rows and 12 distinct ticket IDs**. In ticket-ID order 1001
through 1012, latest event IDs are **5002, 5004, 5007, 5008, 5010, 5012,
5013, 5015, 5016, 5018, 5019, 5021**. Ticket 1004 retains event 5008,
and ticket 1009 retains event 5016 despite their missing assignees.
All 12 have an event in this fixture; LEFT JOIN also protects a future
zero-event ticket, which would have NULL event columns.

### Your Work: Save an Active-Only Latest-Event View

Adapt the worked query to the **7 active tickets** and create
`metro_support.active_ticket_latest_event`. Its columns, in order,
must be **ticket_id, status, event_id, event_type, event_at**. Keep the
window ordering and LEFT JOIN. Apply the active-status condition to
tickets, not to an assignee join or an event's historical status.

Your file should include the view definition, count/distinct-ID checks,
and a query showing **1004 and 1009** remain. Explain the deterministic
tie rule and why it does not prove which tied event happened later.
Independently inspect one multi-event active ticket's original history
and compare its selected event. Explain what would happen to a ticket
with no events if an ON condition `e.rn = 1` moved into WHERE.
Query a view with your own ORDER BY; do not rely on storage order.

Submit **one SQL file**, `week_04_latest_event.sql`. No `source_channel`,
Week 3 view, or additional report is required. The completed active-only
view definition is intentionally not supplied.

In [ ]:
latest_event_sql = """
-- TODO: Adapt the all-ticket example into the required active-only view.
-- TODO: Include count, distinct-ID, and 1004/1009 checks.
-- TODO: Check one multi-event active ticket against its original history.
-- TODO: Explain ties and preserving zero-event tickets in SQL comments.
"""
run_sql(latest_event_sql)

After creating your view, enable the following checks. Expect **7 and
7**, then exactly **1004 / new / 5008** and **1009 / new / 5016**. Matching
row and distinct-ID counts help reveal duplicate ticket rows; the known
IDs check guards against silently losing unassigned tickets.

In [ ]:
CHECK_MY_LATEST_VIEW = False
if CHECK_MY_LATEST_VIEW:
    run_sql("""
    SELECT count(*) AS rows, count(DISTINCT ticket_id) AS distinct_tickets
    FROM metro_support.active_ticket_latest_event;
    SELECT ticket_id, status, event_id, event_type, event_at
    FROM metro_support.active_ticket_latest_event
    WHERE ticket_id IN (1004, 1009)
    ORDER BY ticket_id;
    """)

### Optional: Download Your SQL, Not the Notebook

Finish `latest_event_sql` above, including your SQL comments and checks.
This cell saves **exactly that string**, not every cell you ran and
not your query output. Review the printed text before submitting.
Set the switch to `True` when ready; it writes `week_04_latest_event.sql` in the
runtime and opens Colab's download prompt. On local Jupyter it prints
the file's location. Repeating the export replaces that same file.
Downloading is not submitting; upload the SQL file to Brightspace.

In [ ]:
EXPORT_SQL = False  # Change to True only after finishing the workspace.
if EXPORT_SQL:
    if "-- TODO" in latest_event_sql:
        raise ValueError("Replace the TODO placeholders with your own work first.")
    print(latest_event_sql)  # Preview exactly what will be in the file.
    sql_file = Path("week_04_latest_event.sql")
    sql_file.write_text(latest_event_sql.strip() + "\n", encoding="utf-8")
    if IN_COLAB:
        from google.colab import files
        files.download(str(sql_file))
    else:
        print("Saved:", sql_file.resolve())

## Finish: Remove Only This Notebook's Practice Database

**Download your two SQL files first. Do not run this between days.**
This removes only the uniquely named database created by this run.
In Colab it also stops the service started above, so finish other
PostgreSQL work in this same runtime first. It never stops your
computer's preexisting PostgreSQL service. Saved SQL files are kept.

To start over, run cleanup, then rerun from the setup cells. If the
Python kernel restarts and loses its variables, do not guess a database
name to drop. A new run creates a different practice database. Colab
discards runtime storage eventually; a local operator can inspect and
remove an old practice database separately after confirming ownership.

In [ ]:
if "PRACTICE_DB" in globals():
    assert PRACTICE_DB == "cst4714_week04_" + run_id
    subprocess.run(
        PG_PREFIX + ["dropdb"] + PG_ARGS
        + ["--maintenance-db=postgres", "--if-exists", PRACTICE_DB],
        check=True,
    )
    print("Removed:", PRACTICE_DB)
    del PRACTICE_DB
    globals().pop("FIXTURE_LOADED", None)
if globals().get("COLAB_SERVICE_STARTED", False):
    subprocess.run(["service", "postgresql", "stop"], check=True)
    COLAB_SERVICE_STARTED = False
print("Cleanup complete. A local computer's server is left running.")

**License:** prose CC BY-NC-SA 4.0; code MIT; synthetic data CC0.